In [0]:
# 1 - Criação do Database da camada Gold:

"""
Descrição geral:

- Import do módulo necessário e criação do Database na Gold Layer.
- Uso de prefixos explícitos, visando evitar ambiguidades entre tabelas
de mesmo nome em diferentes databases.
"""

from pyspark.sql import functions as F

# Valição de Idempotência e consolidação do Database
spark.sql("""
    CREATE DATABASE IF NOT EXISTS gold 
    COMMENT 'Camada Gold — Data Marts consolidados para análise de negócio'
""")

print("Imports concluídos.")
print("Banco de dados 'gold' criado")
print()
print("Databases disponíveis:")
spark.sql("SHOW DATABASES").show(truncate=False)

Imports concluídos.
Banco de dados 'gold' criado

Databases disponíveis:
+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|olist_ecommerce   |
|silver            |
+------------------+



In [0]:
# 2 - Construção da Tabela 'gold.fat_vendas_comercial':

"""
Descrição geral do tratamento aplicado:

Passo 1 - Join entre 'fat_itens_pedidos' e 'dim_produtos':
- LEFT JOIN por 'id_produto' traz a categoria de cada item vendido.
- Itens sem correspondência em 'dim_produtos' (produtos removidos do
catálogo) ficam com 'categoria_produto' = 'null' e são mantidos.

Passo 2 - Join com 'fat_pedido_total':
- LEFT JOIN por 'id_pedido' traz 'data_pedido' e 'valor_total_pago_usd'.
- Usa-se 'fat_pedido_total' como fonte de 'data_pedido' pois ela já
consolida o valor pago por pedido e a cotação do dólar aplicada.
- A receita em USD vem de 'valor_total_pago_usd' dividida
proporcionalmente por item via sum() no groupBy.

Passo 3 - Extração de ano e mês:
- F.year() e F.month() extraem as partes da data diretamente.
- Granularidade (Ano x Mês).

Passo 4 - Agregação com as 7 métricas:
- 'total_pedidos': countDistinct(id_pedido) conta pedidos únicos.
. Observaçao: Essa análise é necessária, pois um mesmo pedido com 3 itens conta como 1 pedido, não como 3.
- 'qtd_itens_vendidos': count(id_item) conta cada item individualmente.
- 'receita_total_brl': sum(preco_BRL) soma o preço de cada item.
. Observação: Usa-se 'preco_BRL' de 'fat_itens_pedidos' (preço por item) e não
'valor_total_pago_brl' de 'fat_pedido_total' (total do pedido), pois o que se quer é a receita agregada por categoria e não por pedido.
- 'receita_total_usd': sum(valor_total_pago_usd) agrega o valor em USD.
- 'ticket_medio_brl': 'receita_total_brl'/'total_pedidos' (Valor médio por pedido dentro de cada combinação de ano/mês/categoria).

Passo 5 - Arredondamento obrigatório para 2 casas decimais:
- Aplicado em todos os valores financeiros: 'receita_total_brl', 'receita_total_usd' e 'ticket_medio_brl'.

- A gravação é feita em modo overwrite

"""

# Leitura das tabelas da camada Silver necessárias:
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_total = spark.table("silver.fat_pedido_total")

# Passo 1: Join Itens x Produtos - Traz 'categoria_produto':
df_itens_com_categoria = (
    df_itens
    .join(
        df_produtos.select("id_produto", "categoria_produto"),
        on="id_produto",
        how="left"
    )
)

# Passo 2: Join com 'fat_pedido_total' — Traz 'data_pedido' e valor em USD:
df_base = (
    df_itens_com_categoria
    .join(
        df_total.select("id_pedido", "data_pedido", "valor_total_pago_usd"),
        on="id_pedido",
        how="left"
    )
)

# Passo 3 e 4: Extração de Ano/Mês e agregação das 7 métricas:
df_vendas_comercial = (
    df_base
    .withColumn("ano_venda", F.year(F.col("data_pedido")))
    .withColumn("mes_venda", F.month(F.col("data_pedido")))
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        # 'total_pedidos': Pedidos únicos
        F.countDistinct("id_pedido").alias("total_pedidos"),
        # 'qtd_itens_vendidos': Contagem absoluta de itens
        F.count("id_item").alias("qtd_itens_vendidos"),
        # 'receita_total_brl': Soma dos preços individuais de cada item
        F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
        # 'receita_total_usd': Soma do valor convertido em USD por pedido
        F.round(F.sum("valor_total_pago_usd"), 2).alias("receita_total_usd"),
    )

    # Passo 5: 'ticket_medio_brl' calculado após a agregação
    # Divisão de 'receita_total_brl' pelo número de pedidos únicos
    .withColumn(
        "ticket_medio_brl",
        F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2)
    )

    # Ordenação para a leitura analítica
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

# Gravação na Gold:
(
    df_vendas_comercial.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("gold.fat_vendas_comercial")
)

total = spark.table("gold.fat_vendas_comercial").count()
print(f"gold.fat_vendas_comercial criada: {total:,} combinações ano/mês/categoria")
print()
print("Amostra:")
display(spark.table("gold.fat_vendas_comercial").limit(10))

gold.fat_vendas_comercial criada: 1,283 combinações ano/mês/categoria

Amostra:


ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2016,9,beleza_saude,1,3,134.97,null,134.97
2016,9,moveis_decoracao,1,2,72.89,84.02,72.89
2016,9,telefonia,1,1,59.5,22.94,59.5
2016,10,null,2,2,65.89,29.81,32.95
2016,10,alimentos,1,1,79.9,29.96,79.9
2016,10,audio,2,2,156.99,56.97,78.5
2016,10,automotivo,11,12,1833.25,842.85,166.66
2016,10,bebes,11,14,1630.16,727.09,148.2
2016,10,beleza_saude,44,48,4552.51,1883.69,103.47
2016,10,brinquedos,25,27,4465.09,1760.28,178.6


In [0]:
# 3 - Top 5 Produtos mais vendidos:

"""
Descrição geral:

O Ranking não é gravado em tabela,mas apenas exibido via display()

Origem: 'fat_itens_pedidos' (Uma linha por item vendido) + 'dim_produtos' (Para trazer 'nome_produto' e 'categoria_produto').
- Agrupamos por id_produto e contamos o número de ocorrências cada linha em fat_itens_pedidos representa uma unidade vendida.
- Ordenação: 'quantidade_vendida' de forma decrescente (Maior -> Menor).
- Limite: Top 5.
"""

df_top5_mais = (
    spark.table("silver.fat_itens_pedidos")
    .groupBy("id_produto")
    .agg(
        F.count("id_item").alias("quantidade_vendida")
    )
    # Join com 'dim_produtos' para trazer Nome e Categoria:
    .join(
        spark.table("silver.dim_produtos").select(
            "id_produto", "nome_produto", "categoria_produto"
        ),
        on="id_produto",
        how="left"
    )
    # Selecionando apenas as colunas necessárias:
    .select("nome_produto", "categoria_produto", "quantidade_vendida")
    .orderBy(F.col("quantidade_vendida").desc())
    .limit(5)
)

# Exibição do DataFrame:
print("Top 5 Produtos Mais Vendidos:")
display(df_top5_mais)

Top 5 Produtos Mais Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Estante de Livros Luxo,moveis_decoracao,527
Cobertor Cinza,cama_mesa_banho,488
Cortador de Grama Branco,ferramentas_jardim,484
Kit de Ferramentas Ultra,ferramentas_jardim,392
Kit de Ferramentas Master,ferramentas_jardim,388


In [0]:
# 4 - Top 5 Produtos menos vendidos:

"""
Descrição geral:

Basicamente, é a mesma lógica do Top 5 mais vendidos, mas com duas diferenças:
- Ordenação ascendente (Menor -> Maior) para trazer os produtos com menor saída no histórico.
- Produtos com 'quantidade_vendida' = 1 são completamente válidos, pois indicam produtos vendidos apenas uma vez em todo o período.

Observação:
- Não foi necessário filtrar produtos com estoque zerado ou inativos, pois a base 'fat_itens_pedidos' já reflete apenas
itens efetivamente vendidos, ou seja, que geraram um pedido.
- Produtos nunca vendidos não aparecem em 'fat_itens_pedidos' e, por isso, não entram nesse ranking.
"""

df_top5_menos = (
    spark.table("silver.fat_itens_pedidos")
    .groupBy("id_produto")
    .agg(
        F.count("id_item").alias("quantidade_vendida")
    )
    # Join com 'dim_produtos' para trazer Nome e Categoria:
    .join(
        spark.table("silver.dim_produtos").select(
            "id_produto", "nome_produto", "categoria_produto"
        ),
        on="id_produto",
        how="left"
    )
    # Selecionando apenas as colunas relevantes:
    .select("nome_produto", "categoria_produto", "quantidade_vendida")
    # Ordenação ascendente (Menor -> Maior):
    .orderBy(F.col("quantidade_vendida").asc())
    .limit(5)
)

# Exibição do DataFrame:
print("Top 5 Produtos Menos Vendidos:")
display(df_top5_menos)

Top 5 Produtos Menos Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo Cinza,beleza_saude,1
Cadeira de Escritório,moveis_decoracao,1
Estante de Livros Cinza,moveis_decoracao,1
Edredom Avançado,cama_mesa_banho,1
Creme Anti-idade Prata,beleza_saude,1


In [0]:
# 5 - Construção da Tabela gold.fat_avaliacoes_clientes:

"""
Descrição geral do tratamento aplicado:

Passo 1 - Join Avaliações x Itens:
- 'fat_avaliacoes_pedidos' está no nível de pedido (uma avaliação por pedido).
- 'fat_itens_pedidos' está no nível de item (um pedido pode ter N itens).
- O join por 'id_pedido' pode gerar múltiplas linhas por avaliação quando um pedido tem mais de um item ou vendedor diferente.
- Esse comportamento é esperado, pois uma avaliação de pedido com 2 itens de vendedores distintos deve ser contabilizada para ambos os vendedores.

Passo 2 - Join com 'dim_produtos':
- Traz 'categoria_produto' para a granularidade.
- LEFT JOIN: Itens sem produto cadastrado mantêm categoria = 'null'.

Passo 3 - Join com 'dim_vendedores':
- Traz 'nome_vendedor' e 'estado' para a granularidade.
- LEFT JOIN: Itens sem vendedor cadastrado mantêm esses campos = 'null'.

Passo 4 - Agregação com as 5 métricas:
- 'total_avaliacoes': count(*) (Conta todas as avaliações do grupo).
- 'avaliacao_media': avg(nota_avaliacao) (Média matemática das notas).
- 'total_avaliacoes_positivas': Soma condicional com F.when para notas >= 4.
. Observação: F.when(condição, 1).otherwise(0) cria um flag por linha, sum() soma os flags.
- 'total_avaliacoes_negativas': Mesma lógica para notas <= 2.
- 'percentual_satisfacao': É o cálculo [(positivas/total) * 100] - Percentual de satisfação que é calculado após a agregação 
para usar os valores já consolidados.
. Observação: F.when(total > 0, ...) evita divisão por zero em grupos sem avaliações.

A gravação é feita em modo Overwrite.
"""

from pyspark.sql import functions as F

# Leitura das tabelas da camada Silver necessárias:
df_avaliacoes = spark.table("silver.fat_avaliacoes_pedidos")
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_vendedores = spark.table("silver.dim_vendedores")

# Passos realizados:

# Passo 1: Join Avaliações x Itens - Conecta pedido aos seus produtos e vendedores
# Passo 2: Join com 'dim_produtos' - Traz 'categoria_produto'
# Passo 3: Join com 'dim_vendedores' - Traz 'nome_vendedor' e 'estado'
df_base = (
    df_avaliacoes
    .select("id_pedido", "nota_avaliacao")
    .join(
        df_itens.select("id_pedido", "id_produto", "id_vendedor"),
        on="id_pedido",
        how="left"
    )
    .join(
        df_produtos.select("id_produto", "categoria_produto"),
        on="id_produto",
        how="left"
    )
    .join(
        df_vendedores.select("id_vendedor", "nome_vendedor", "estado"),
        on="id_vendedor",
        how="left"
    )
)

# Passo 4: Agregação com as 5 métricas
df_avaliacoes_clientes = (
    df_base
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        # Contagem absoluta de avaliações no grupo:
        F.count("*").alias("total_avaliacoes"),
        # Média matemática das notas (Arredondamento para 2 casas):
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        # Avaliações positivas: notas >= 4:
        F.sum(
            F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)
        ).alias("total_avaliacoes_positivas"),
        # Avaliações negativas: notas <= 2:
        F.sum(
            F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)
        ).alias("total_avaliacoes_negativas"),
    )

    # Calculando 'percentual_satisfacao'após a agregação:
    .withColumn(
        "percentual_satisfacao",
        F.round(
            F.when(
                F.col("total_avaliacoes") > 0,
                (F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100
            ).otherwise(0),
            2
        )
    )
    .orderBy("categoria_produto", "nome_vendedor")
)

# Gravação na Gold:
(
    df_avaliacoes_clientes.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("gold.fat_avaliacoes_clientes")
)

total = spark.table("gold.fat_avaliacoes_clientes").count()
print(f"gold.fat_avaliacoes_clientes criada: {total:,} combinações categoria/vendedor/estado")
print()
print("Amostra:")
display(spark.table("gold.fat_avaliacoes_clientes").limit(10))

gold.fat_avaliacoes_clientes criada: 6,584 combinações categoria/vendedor/estado

Amostra:


categoria_produto,nome_vendedor,estado,total_avaliacoes,avaliacao_media,total_avaliacoes_positivas,total_avaliacoes_negativas,percentual_satisfacao
null,null,null,3457,1.72,104,603,3.01
null,Agatha Moura,PR,3,4.67,3,0,100.0
null,Alana Araújo,SP,1,1.0,0,1,0.0
null,Alexandre Aparecida,SP,2,4.5,2,0,100.0
null,Alexia Garcia,SP,1,3.0,0,0,0.0
null,Alexia Nascimento,RS,2,5.0,2,0,100.0
null,Alexia Rodrigues,SC,2,1.0,0,2,0.0
null,Alice Aparecida,RS,3,4.67,3,0,100.0
null,Alice da Cunha,SP,9,4.67,9,0,100.0
null,Alícia Carvalho,SP,7,3.86,6,1,85.71


In [0]:
# 6 - Produto mais bem avaliado:

"""
Descrição geral:

Agrupa-se por 'categoria_produto' (Que é o proxy de produto na granularidade em 'fat_avaliacoes_clientes') e se aplica o critério de
ordenação composto:
- Critério 1: 'avaliacao_media' em ordenação descendente (A maior nota vem primeiro).
- Critério 2: 'total_avaliacoes' em ordenação descendente e , em caso de empate na nota, o produto com mais avaliações (Maior volume) é 
considerado mais confiável estatisticamente e sobe no ranking.

A tabela 'fat_avaliacoes_clientes', que já está agregada, foi usada como fonte, reagrupando por 'categoria_produto' para consolidar todos os
vendedores de uma mesma categoria em uma única linha.
"""

df_produto_mais_avaliado = (
    spark.table("gold.fat_avaliacoes_clientes")
    # Reagrupando os vendedores por categoria:
    .groupBy("categoria_produto")
    .agg(
        F.round(F.avg("avaliacao_media"), 2).alias("avaliacao_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes"),
    )
    # Aplicação dos critérios:
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
)

print("Produto (Categoria) Mais Bem Avaliado:")
display(df_produto_mais_avaliado)

Produto (Categoria) Mais Bem Avaliado:


categoria_produto,avaliacao_media,total_avaliacoes
cds_dvds_musicais,4.64,14


In [0]:
# 7 - Produto menos bem avaliado:

"""
Descrição geral:

Resumidamente, é a mesma lógica do produto mais bem avaliado, apenas com a ordenação invertida no primeiro critério:
- Critério 1: 'avaliacao_media' em ordenação ascendente (A menor nota vem primeiro).
- Critério 2: 'total_avaliacoes' em ordenação descendente e, em caso de empate na nota, o produto com mais avaliações negativas (Maior volume) é mais representativo e sobe no ranking de piores.
"""

df_produto_menos_avaliado = (
    spark.table("gold.fat_avaliacoes_clientes")
    # Reagrupamento por categoria:
    .groupBy("categoria_produto")
    .agg(
        F.round(F.avg("avaliacao_media"), 2).alias("avaliacao_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes"),
    )
    # Aplicação dos critérios:
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
)

print("Produto (Categoria) Menos Bem Avaliado:")
display(df_produto_menos_avaliado)

Produto (Categoria) Menos Bem Avaliado:


categoria_produto,avaliacao_media,total_avaliacoes
seguros_e_servicos,2.5,2


In [0]:
# 8 - Vendedor mais bem avaliado:

"""
Descrição geral:

Agrupa-se por 'nome_vendedor' e 'estado', conforme a granularidade de 'dim_vendedores', e se aplica o critério de ordenação composto para o caso.
- Critério 1: 'avaliacao_media' em ordem descendente (A maior nota vem primeiro).
- Critério 2: 'total_avaliacoes' em ordem descendente e para o caso de empate na nota, o vendedor com mais avaliações (Volume maior) é mais representativo estatisticamente e sobe no ranking.

. Observação: Foi incluído  o 'estado' na exibição, pois ele faz parte da granularidade definida para 'fat_avaliacoes_clientes'.
"""

df_vendedor_mais_avaliado = (
    spark.table("gold.fat_avaliacoes_clientes")
    # Reagrupando por vendedor para consolidar as categorias:
    .groupBy("nome_vendedor", "estado")
    .agg(
        F.round(F.avg("avaliacao_media"), 2).alias("avaliacao_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes"),
    )
    # Aplicação dos critérios:
    .orderBy(
        F.col("avaliacao_media").desc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
)

print("Vendedor Mais Bem Avaliado:")
display(df_vendedor_mais_avaliado)

Vendedor Mais Bem Avaliado:


nome_vendedor,estado,avaliacao_media,total_avaliacoes
Luiz Otávio Abreu,PR,5.0,34


In [0]:
# 9 - Vendedor Menos Bem Avaliado:

"""
Descrição geral:

É a mesma lógica do vendedor mais bem avaliado, mas com ordenação invertida no primeiro critério:
- Critério 1: 'avaliacao_media' em ordem ascendente (A menor nota vem primeiro).
- Critério 2: 'total_avaliacoes' em ordem descendente e para o caso de empate na nota, o vendedor com mais avaliações negativas, ou seja, mais volume, é mais representativo e sobe no ranking de piores.
"""

df_vendedor_menos_avaliado = (
    spark.table("gold.fat_avaliacoes_clientes")
    # Reagrupamento para consolidar as categorias:
    .groupBy("nome_vendedor", "estado")
    .agg(
        F.round(F.avg("avaliacao_media"), 2).alias("avaliacao_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes"),
    )
    # Aplicação dos critérios:
    .orderBy(
        F.col("avaliacao_media").asc(),
        F.col("total_avaliacoes").desc()
    )
    .limit(1)
)

print("Vendedor Menos Bem Avaliado:")
display(df_vendedor_menos_avaliado)

Vendedor Menos Bem Avaliado:


nome_vendedor,estado,avaliacao_media,total_avaliacoes
Sra. Fernanda Santos,SP,1.0,8
